# Dataset merging

In [ ]:
import pandas as pd
import os
import geopandas as gpd
from shapely import make_valid

from utils.config import find_repo_root

wd = find_repo_root()
os.chdir(wd)

## Sub-regional institutional raw data

In [ ]:
# Define datasets to load
datasets = {
  'camels_cl': ('data/CAMELS_CL/CAMELS_CL_metadata.csv',
                'data/CAMELS_CL/CAMELS_CL_daily_1950_2025.csv',
                'data/CAMELS_CL/basins_CAMELS_CL.gpkg'),
  'pmetobs':   ('data/PMET_OBS/Q_PMETobs_v11_metadata.csv', 
                'data/PMET_OBS/Q_PMETobs_1950_2025_v11d.csv', 
                'data/PMET_OBS/basins_PMETobs_v11.gpkg'),
  'snhi_arg':  ('data/SNHI_ARG/SNHI_metadata.csv', 
                'data/SNHI_ARG/SNHI_daily_1950_2024.csv', 
                'data/SNHI_ARG/basins_SNHI.gpkg'),
  'senamhi':   ('data/SENAMHI_PERU/SENAMHI_metadata.csv', 
                'data/SENAMHI_PERU/SENAMHI_daily_1950_2024.csv', 
                'data/SENAMHI_PERU/basins_SENAMHI.gpkg')}

# Load all datasets
for name, (meta_file, data_file, shape_file) in datasets.items():
    
    globals()[f'metadata_{name}'] = pd.read_csv(meta_file, index_col=0)
    globals()[f'data_{name}'] = pd.read_csv(data_file, index_col=0, parse_dates=["date"]).loc['1950':'2024']
    globals()[f'shape_{name}'] = gpd.read_file(shape_file)[["gauge_id", "geometry"]].set_index("gauge_id")

## Preprocessing / homogenize format

In [ ]:
# Standardize gauge_id format
for meta, shape, data in [(metadata_camels_cl, shape_camels_cl, data_camels_cl)]:
    meta.index     = ['X' + str(i).zfill(8) for i in meta.index]
    shape.index    = ['X' + str(i).zfill(8) for i in shape.index]
    data.columns   = ['X' + str(i).zfill(8) for i in data.columns]

# Add metadata attributes
metadata_snhi_arg["gauge_name"] = metadata_snhi_arg.river + " " + metadata_snhi_arg.place
metadata_snhi_arg[["institution",  "country", "dataset"]]  = ["SNHI",    "Argentina", "Andean-GC (self-produced)"]
metadata_camels_cl[["institution", "country", "dataset"]]  = ["DGA",     "Chile",     "CAMELS-cl (updated)"]
metadata_senamhi[["institution",   "country", "dataset"]]  = ["SENAMHI", "Peru",      "Andean-GC (self-produced)"]
metadata_pmetobs[["institution",   "country", "dataset"]]  = ["PMETobs", "Multiple",  "PMETobs v11 (updated)"]

# Keep only essential columns
attributes = ["gauge_lat", "gauge_lon", "gauge_name", "institution", "country", "dataset"]
for meta in [metadata_snhi_arg, metadata_camels_cl, metadata_pmetobs, metadata_senamhi]:
    meta.drop(columns=meta.columns.difference(attributes), inplace=True)

# Fix invalid geometries
shape_camels_cl["geometry"] = make_valid(shape_camels_cl.geometry)

## Concatenation and saving

In [ ]:
# concatenate
AndeanGC_metadata = pd.concat([metadata_snhi_arg, metadata_camels_cl, metadata_pmetobs, metadata_senamhi]).reset_index()
AndeanGC_metadata = AndeanGC_metadata.drop_duplicates(subset=["index"], keep="last")
AndeanGC_metadata = AndeanGC_metadata.rename(columns={"index": "gauge_id"}).set_index("gauge_id")
AndeanGC_metadata["gauge_name"] = AndeanGC_metadata["gauge_name"].str.replace("_", " ").str.title()

# Filter conditions for each dataset (gives priority to PMET)
arg_filter = (AndeanGC_metadata.dataset == "Andean-GC (self-produced)") & (AndeanGC_metadata.country == "Argentina")
peru_filter = (AndeanGC_metadata.dataset == "Andean-GC (self-produced)") & (AndeanGC_metadata.country == "Peru")
chile_filter = AndeanGC_metadata.dataset == "CAMELS-cl (updated)"

AndeanGC_data = pd.concat([
    data_snhi_arg[AndeanGC_metadata[arg_filter].index],
    data_senamhi[AndeanGC_metadata[peru_filter].index],
    data_camels_cl[AndeanGC_metadata[chile_filter].index],
    data_pmetobs], axis=1)

AndeanGC_shape = pd.concat([
    shape_snhi_arg.loc[AndeanGC_metadata[arg_filter].index],
    shape_senamhi.loc[AndeanGC_metadata[peru_filter].index],
    shape_camels_cl.loc[AndeanGC_metadata[chile_filter].index],
    shape_pmetobs])
AndeanGC_shape = pd.concat([AndeanGC_metadata, AndeanGC_shape], axis=1)
AndeanGC_shape = gpd.GeoDataFrame(AndeanGC_shape, geometry='geometry')

In [ ]:
# Save processed data
AndeanGC_metadata.to_csv('dataset/AndeanGC_metadata.csv')
AndeanGC_data.to_csv('dataset/AndeanGC_data_1950_2024.csv')
AndeanGC_shape.to_file('dataset/AndeanGC_shape.gpkg')